# Credit Default Prediction — Homework #2
## Decision Tree · Random Forest · Gradient Boosting · **Custom GBM ⭐**

---
**Author:** HSE Business Analytics, 2026  
**Dataset:** `train.csv` (7,500 × 17), `test.csv` (2,500 × 16)  
**Target:** `Credit Default` = 1 (binary classification)

### Pipeline
| Step | Description |
|------|-------------|
| 1 | EDA — statistics, missing values, class imbalance |
| 2 | Outlier Analysis — all numeric features, IQR method |
| 3 | Feature Engineering — DTI, Credit Utilization, log-transforms |
| 4 | Correlation Analysis + Feature Selection (|r| > 0.70) |
| 5 | Preprocessing — encoding, imputation, scaling |
| 6 | Library Models — LR, DT, RF, LightGBM |
| 7 | **Custom GBM ⭐** — hand-written gradient boosting |
| 8 | Comparison Library vs Custom |
| 9 | Predictions for test set |


## 0. Imports & Setup

In [32]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os, glob, time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, roc_auc_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              precision_recall_curve)
import lightgbm as lgb

OUT  = '/mnt/user-data/outputs/hw2_final'
os.makedirs(OUT, exist_ok=True)
SEED = 42
np.random.seed(SEED)
print(f'numpy {np.__version__}  pandas {pd.__version__}  lgbm {lgb.__version__}')
print('All libraries loaded ✓')


numpy 2.0.2  pandas 2.2.2  lgbm 4.6.0
All libraries loaded ✓


## Part 1. EDA — Exploratory Data Analysis

### Key observations:
- **7,500 training rows**, 17 features (12 numeric, 5 categorical)
- **Class imbalance: 71.8% / 28.2%** — moderate, handled with `class_weight='balanced'`
- `Months since last delinquent`: 54.4% missing — **informative absence** (will become `Has_Delinquent`)
- `Annual Income` + `Credit Score`: 20.8% missing — impute with median


In [33]:
train = pd.read_csv('course_project_train.csv')
test  = pd.read_csv('course_project_test.csv')
print(f'Train: {train.shape}   Test: {test.shape}')
print()
print('Missing values:')
miss = train.isnull().sum()
miss_pct = (miss / len(train) * 100).round(2)
print(pd.DataFrame({'count': miss, 'pct_%': miss_pct})
      [pd.DataFrame({'count': miss, 'pct_%': miss_pct})['count'] > 0]
      .sort_values('count', ascending=False))


Train: (7500, 17)   Test: (2500, 16)

Missing values:
                              count  pct_%
Months since last delinquent   4081  54.41
Annual Income                  1557  20.76
Credit Score                   1557  20.76
Years in current job            371   4.95
Bankruptcies                     14   0.19


In [34]:
# Class distribution
vc = train['Credit Default'].value_counts()
print(f'Class balance: {vc[0]} ({vc[0]/len(train)*100:.1f}%) vs {vc[1]} ({vc[1]/len(train)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No Default (0)', 'Default (1)'], vc.values,
               color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+40,
            f'{val}\n({val/len(train)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Class Imbalance — Credit Default', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(vc.values)*1.18)
plt.tight_layout(); plt.savefig(f'{OUT}/01_class_imbalance.png', dpi=120, bbox_inches='tight')
plt.show()


Class balance: 5387 (71.8%) vs 2113 (28.2%)


In [35]:
# Numeric statistics
num_all = train.select_dtypes(include='number').columns.tolist()
train[num_all].describe().T[['mean','std','min','25%','50%','75%','max']].round(2)


,mean,std,min,25%,50%,75%,max
Annual Income,1366391.72,845339.20,164597.0,844341.0,1168386.0,1640137.00,1.014934e+07
Tax Liens,0.03,0.27,0.0,0.0,0.0,0.00,7.000000e+00
Number of Open Accounts,11.13,4.91,2.0,8.0,10.0,14.00,4.300000e+01
Years of Credit History,18.32,7.04,4.0,13.5,17.0,21.80,5.770000e+01
Maximum Open Credit,945153.73,16026216.67,0.0,279229.5,478159.0,793501.50,1.304726e+09
Number of Credit Problems,0.17,0.50,0.0,0.0,0.0,0.00,7.000000e+00
Months since last delinquent,34.69,21.69,0.0,16.0,32.0,50.00,1.180000e+02
Bankruptcies,0.12,0.35,0.0,0.0,0.0,0.00,4.000000e+00
Current Loan Amount,11873177.45,31926122.97,11242.0,180169.0,309573.0,519882.00,1.000000e+08
Current Credit Balance,289833.24,317871.38,0.0,114256.5,209323.0,360406.25,6.506797e+06


In [36]:
# Distributions split by class
num_plot = [c for c in num_all if c != 'Credit Default'][:12]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(num_plot):
    ax = axes[i]
    for cls, clr, lbl in [(0,'#2ecc71','No Default'),(1,'#e74c3c','Default')]:
        ax.hist(train[train['Credit Default']==cls][col].dropna(),
                bins=40, alpha=0.6, color=clr, label=lbl, density=True)
    ax.set_title(col, fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=6)
    if i == 0: ax.legend(fontsize=7)
for j in range(len(num_plot), len(axes)): axes[j].set_visible(False)
fig.suptitle('Feature Distributions by Target Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT}/02_distributions_by_class.png', dpi=110, bbox_inches='tight')
plt.show()


## Part 2. Outlier Analysis — All Numeric Features

**Method:** IQR (Interquartile Range), fence = Q1 − 1.5·IQR and Q3 + 1.5·IQR

### Critical findings:
| Feature | Problem | Action |
|---|---|---|
| `Current Loan Amount` | Placeholder **99999999** — 870 rows (11.6%) | → NaN → log-transform |
| `Credit Score` | Values **> 850** (outside FICO scale 300–850) — 400 rows | → NaN |
| `Maximum Open Credit` | Max = **1.3 billion**, extreme right tail | → log-transform |
| `Number of Credit Problems` | 13.7% outliers by IQR | **kept** — real events, informative |
| `Bankruptcies` | 11% outliers + collinear with NCP | **removed** in feature selection |

> ⚠️ Outliers are **not removed** from the dataset — in credit scoring, extreme values  
> often carry the strongest signal for default prediction.


In [37]:
# Replace placeholder and fix Credit Score
PLACEHOLDER = 99999999
n_ph = (train['Current Loan Amount'] == PLACEHOLDER).sum()
print(f'Placeholder {PLACEHOLDER} in Current Loan Amount: {n_ph} rows ({n_ph/len(train)*100:.1f}%) → NaN')
train['Current Loan Amount'] = train['Current Loan Amount'].replace(PLACEHOLDER, np.nan)
test['Current Loan Amount']  = test['Current Loan Amount'].replace(PLACEHOLDER, np.nan)

n_cs = (train['Credit Score'] > 850).sum()
print(f'Credit Score > 850 (outside FICO 300–850): {n_cs} rows → NaN')
train['Credit Score'] = train['Credit Score'].where(train['Credit Score'] <= 850, np.nan)
test['Credit Score']  = test['Credit Score'].where(test['Credit Score'] <= 850, np.nan)


Placeholder 99999999 in Current Loan Amount: 870 rows (11.6%) → NaN
Credit Score > 850 (outside FICO 300–850): 400 rows → NaN


In [38]:
# Full IQR analysis
num_cols = [c for c in train.select_dtypes(include='number').columns if c != 'Credit Default']
rows = []
for col in num_cols:
    s = train[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = int(((s < lo) | (s > hi)).sum())
    rows.append({'Feature': col, 'Q1': round(Q1,1), 'Q3': round(Q3,1),
                 'Lower Fence': round(lo,1), 'Upper Fence': round(hi,1),
                 'N Outliers': n_out, '% Outliers': round(n_out/len(s)*100, 2),
                 'Min': round(s.min(),1), 'Max': round(s.max(),1)})
ods = pd.DataFrame(rows)
ods.to_csv(f'{OUT}/outlier_summary.csv', index=False)
ods


,Feature,Q1,Q3,Lower Fence,Upper Fence,N Outliers,% Outliers,Min,Max
0,Annual Income,844341.0,1640137.0,-349353.0,2833831.0,287,4.83,164597.0,1.014934e+07
1,Tax Liens,0.0,0.0,0.0,0.0,134,1.79,0.0,7.000000e+00
2,Number of Open Accounts,8.0,14.0,-1.0,23.0,170,2.27,2.0,4.300000e+01
3,Years of Credit History,13.5,21.8,1.0,34.2,228,3.04,4.0,5.770000e+01
4,Maximum Open Credit,279229.5,793501.5,-492178.5,1564909.5,466,6.21,0.0,1.304726e+09
5,Number of Credit Problems,0.0,0.0,0.0,0.0,1031,13.75,0.0,7.000000e+00
6,Months since last delinquent,16.0,50.0,-35.0,101.0,1,0.03,0.0,1.180000e+02
7,Bankruptcies,0.0,0.0,0.0,0.0,826,11.03,0.0,4.000000e+00
8,Current Loan Amount,172744.0,430100.0,-213290.0,816134.0,0,0.00,11242.0,7.890300e+05
9,Current Credit Balance,114256.5,360406.2,-254968.1,729630.9,440,5.87,0.0,6.506797e+06


In [39]:
# Boxplots
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    ax = axes[i]
    ax.boxplot(train[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#4C72B0', alpha=0.7),
               medianprops=dict(color='#e74c3c', linewidth=2),
               flierprops=dict(marker='o', markersize=1.5, alpha=0.3, color='gray'))
    ax.set_title(col, fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=6)
for j in range(len(num_cols), len(axes)): axes[j].set_visible(False)
fig.suptitle('Outlier Analysis — Boxplots (IQR, 1.5×IQR)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT}/03_outliers_boxplot.png', dpi=110, bbox_inches='tight')
plt.show()


## Part 3. Feature Engineering

| New Feature | Formula | Economic Rationale |
|---|---|---|
| `DTI` | Monthly Debt / (Annual Income / 12) | **Debt-to-income ratio** — primary credit risk metric in banking |
| `Credit_Utilization` | Current Credit Balance / Max Open Credit | Limit usage > 70% signals financial stress |
| `Has_Delinquent` | 1 if delinquency record exists | Replaces 54%-NaN continuous feature |
| `Log_Income` | log(1 + Annual Income) | Removes right skew from income distribution |
| `Log_Loan` | log(1 + Current Loan Amount) | Normalises loan amount scale |
| `Log_MaxCred` | log(1 + Maximum Open Credit) | Normalises credit limit (max = 1.3B) |


In [40]:
def feature_engineering(df):
    df = df.copy()
    df['DTI']              = df['Monthly Debt'] / (df['Annual Income'] / 12 + 1)
    moc = df['Maximum Open Credit'].clip(lower=1)
    df['Credit_Utilization'] = (df['Current Credit Balance'] / moc).clip(upper=5)
    df['Has_Delinquent']   = df['Months since last delinquent'].notna().astype(int)
    df['Log_Income']       = np.log1p(df['Annual Income'])
    df['Log_Loan']         = np.log1p(df['Current Loan Amount'])
    df['Log_MaxCred']      = np.log1p(df['Maximum Open Credit'])
    return df

train = feature_engineering(train)
test  = feature_engineering(test)
new_feats = ['DTI','Credit_Utilization','Has_Delinquent','Log_Income','Log_Loan','Log_MaxCred']
print('New feature statistics:')
train[new_feats].describe().T[['mean','std','min','max']].round(4)


New feature statistics:


,mean,std,min,max
DTI,0.1714,0.0793,0.0000,0.3990
Credit_Utilization,0.4805,0.2770,0.0000,5.0000
Has_Delinquent,0.4559,0.4981,0.0000,1.0000
Log_Income,13.9858,0.5224,12.0113,16.1329
Log_Loan,12.4433,0.6758,9.3275,13.5786
Log_MaxCred,12.9548,1.4738,0.0000,20.9893


## Part 4. Correlation Analysis + Feature Selection

### Correlation threshold: |r| > 0.70

**Justification** (Hair et al., 2019 — *Multivariate Data Analysis*):
- At |r| > 0.70, the Variance Inflation Factor (VIF) > 2.04 — substantial multicollinearity
- **Logistic Regression**: inflates coefficient variance, widens confidence intervals
- **Decision Tree**: creates redundant splits, reduces interpretability
- **GBM/RF**: minor effect, but reduces feature importance clarity

### Features removed (replaced by engineered equivalents):


In [41]:
# Encode categoricals for correlation
tc = train.copy()
for c in tc.select_dtypes('object').columns:
    tc[c] = LabelEncoder().fit_transform(tc[c].fillna('Unknown'))
ncols = [c for c in tc.select_dtypes(include='number') if c != 'Credit Default']
corr  = tc[ncols].corr()

# Correlation heatmap (lower triangle only)
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(17, 14))
sns.heatmap(corr, mask=mask,
            cmap=sns.diverging_palette(220, 20, as_cmap=True),
            vmin=-1, vmax=1, center=0,
            annot=True, fmt='.2f', annot_kws={'size': 6.5},
            square=True, linewidths=0.4,
            cbar_kws={'shrink': 0.75, 'label': 'Pearson r'}, ax=ax)
ax.set_title('Correlation Matrix (lower triangle, Pearson r)\nRemoval threshold: |r| > 0.70',
             fontsize=12, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=7.5)
plt.yticks(fontsize=7.5)
plt.tight_layout()
plt.savefig(f'{OUT}/04_correlation_matrix.png', dpi=130, bbox_inches='tight')
plt.show()


In [42]:
THRESHOLD_CORR = 0.70
print(f'High-correlation pairs (|r| > {THRESHOLD_CORR}):')
for i in range(len(corr.columns)):
    for j in range(i):
        r = corr.iloc[i, j]
        if abs(r) > THRESHOLD_CORR:
            print(f'  {corr.columns[i]:<30} <-> {corr.columns[j]:<30}  r = {r:.3f}')

DROP_COLS = ['Current Credit Balance', 'Bankruptcies', 'Months since last delinquent',
             'Annual Income', 'Current Loan Amount', 'Maximum Open Credit']
print(f'\nFeatures to drop: {DROP_COLS}')
print('Reason: replaced by engineered equivalents (Credit_Utilization, Has_Delinquent, Log_*)')


High-correlation pairs (|r| > 0.7):
  Bankruptcies                   <-> Number of Credit Problems       r = 0.731
  Log_Income                     <-> Annual Income                   r = 0.903
  Log_Loan                       <-> Current Loan Amount             r = 0.925

Features to drop: ['Current Credit Balance', 'Bankruptcies', 'Months since last delinquent', 'Annual Income', 'Current Loan Amount', 'Maximum Open Credit']
Reason: replaced by engineered equivalents (Credit_Utilization, Has_Delinquent, Log_*)


## Part 5. Preprocessing

**Steps:**
1. Drop collinear/replaced features
2. Ordinal encoding `Years in current job` (< 1 year → 0, 10+ years → 10)
3. Label Encoding for `Home Ownership`, `Purpose`, `Term`
4. Median imputation for numeric missing values
5. `StandardScaler` for Logistic Regression (tree-based models don't require it)


In [43]:
TARGET = 'Credit Default'
JOB_MAP = {'< 1 year':0,'1 year':1,'2 years':2,'3 years':3,'4 years':4,
           '5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10}

def preprocess(df, target=None, encoders=None, drop_cols=None):
    df = df.copy()
    if drop_cols:
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    if 'Years in current job' in df.columns:
        df['Years in current job'] = df['Years in current job'].map(JOB_MAP)
    cat_cols = df.select_dtypes('object').columns.tolist()
    fit_mode = encoders is None
    if fit_mode: encoders = {}
    for c in cat_cols:
        vals = df[c].fillna('Unknown').astype(str)
        if fit_mode:
            le = LabelEncoder(); le.fit(vals); encoders[c] = le
        else:
            le = encoders[c]
            vals = vals.apply(lambda x: x if x in set(le.classes_) else le.classes_[0])
        df[c] = le.transform(vals)
    if target and target in df.columns:
        y = df[target]; X = df.drop(columns=[target])
    else:
        y = None; X = df
    num = X.select_dtypes(include='number').columns.tolist()
    if fit_mode:
        imp = SimpleImputer(strategy='median')
        X[num] = imp.fit_transform(X[num]); encoders['_imp'] = imp
    else:
        X[num] = encoders['_imp'].transform(X[num])
    return (X, y, encoders) if fit_mode else (X, y)

X_all, y_all, enc = preprocess(train, TARGET, drop_cols=DROP_COLS)
X_te, _           = preprocess(test,  None, enc, drop_cols=DROP_COLS)
print(f'X_all: {X_all.shape}   X_te: {X_te.shape}')
print(f'Features: {list(X_all.columns)}')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all)
sc = StandardScaler()
X_tr_s  = pd.DataFrame(sc.fit_transform(X_tr),  columns=X_tr.columns)
X_val_s = pd.DataFrame(sc.transform(X_val),     columns=X_val.columns)
X_te_s  = pd.DataFrame(sc.transform(X_te),      columns=X_te.columns)
pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f'Train: {X_tr.shape}   Val: {X_val.shape}   pos_weight={pos_weight:.2f}')


X_all: (7500, 16)   X_te: (2500, 16)
Features: ['Home Ownership', 'Years in current job', 'Tax Liens', 'Number of Open Accounts', 'Years of Credit History', 'Number of Credit Problems', 'Purpose', 'Term', 'Monthly Debt', 'Credit Score', 'DTI', 'Credit_Utilization', 'Has_Delinquent', 'Log_Income', 'Log_Loan', 'Log_MaxCred']
Train: (6000, 16)   Val: (1500, 16)   pos_weight=2.55


## Helper: Evaluation Function + Threshold Tuning

In [44]:
def best_threshold(y_true, prob):
    prec, rec, thr = precision_recall_curve(y_true, prob)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    idx = np.argmax(f1s)
    return float(thr[idx]) if idx < len(thr) else 0.5

def evaluate_model(name, prob, y_true):
    t    = best_threshold(y_true, prob)
    pred = (prob >= t).astype(int)
    pred5 = (prob >= 0.5).astype(int)
    f1t  = f1_score(y_true, pred)
    f1_5 = f1_score(y_true, pred5)
    auc  = roc_auc_score(y_true, prob)
    gini = 2 * auc - 1
    print(f'  {name}')
    print(f'  threshold=0.500 → F1={f1_5:.4f}')
    print(f'  threshold={t:.3f} → F1={f1t:.4f} (optimal)')
    print(f'  AUC-ROC={auc:.4f}   Gini={gini:.4f}')
    print(classification_report(y_true, pred, target_names=['No Default','Default'], digits=4))
    return {'name':name,'f1_default':round(f1_5,4),'f1_tuned':round(f1t,4),
            'threshold':round(t,3),'auc':round(auc,4),'gini':round(gini,4)}

results = []
print('Evaluation function defined ✓')


Evaluation function defined ✓


## Part 6. Library Models

In [45]:
# Logistic Regression
print('=== Logistic Regression ===')
t0 = time.time()
lr = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=SEED)
lr.fit(X_tr_s, y_tr)
r = evaluate_model('LR (library)', lr.predict_proba(X_val_s)[:,1], y_val)
r.update({'train_sec': round(time.time()-t0,2), 'model':lr, 'scaled':True})
results.append(r)


=== Logistic Regression ===
  LR (library)
  threshold=0.500 → F1=0.4586
  threshold=0.400 → F1=0.4774 (optimal)
  AUC-ROC=0.6503   Gini=0.3006
              precision    recall  f1-score   support

  No Default     0.8395    0.3593    0.5033      1077
     Default     0.3359    0.8251    0.4774       423

    accuracy                         0.4907      1500
   macro avg     0.5877    0.5922    0.4903      1500
weighted avg     0.6975    0.4907    0.4960      1500



In [46]:
# Decision Tree
print('=== Decision Tree ===')
t0 = time.time()
dt = DecisionTreeClassifier(max_depth=7, min_samples_leaf=20,
                             class_weight='balanced', random_state=SEED)
dt.fit(X_tr, y_tr)
r = evaluate_model('DT (library)', dt.predict_proba(X_val)[:,1], y_val)
r.update({'train_sec': round(time.time()-t0,2), 'model':dt, 'scaled':False})
results.append(r)


=== Decision Tree ===
  DT (library)
  threshold=0.500 → F1=0.4821
  threshold=0.369 → F1=0.4975 (optimal)
  AUC-ROC=0.7002   Gini=0.4004
              precision    recall  f1-score   support

  No Default     0.8594    0.4085    0.5538      1077
     Default     0.3553    0.8298    0.4975       423

    accuracy                         0.5273      1500
   macro avg     0.6073    0.6192    0.5257      1500
weighted avg     0.7172    0.5273    0.5379      1500



In [47]:
# Random Forest
print('=== Random Forest ===')
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=15,
                              class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_tr, y_tr)
r = evaluate_model('RF (library)', rf.predict_proba(X_val)[:,1], y_val)
r.update({'train_sec': round(time.time()-t0,2), 'model':rf, 'scaled':False})
results.append(r)


=== Random Forest ===
  RF (library)
  threshold=0.500 → F1=0.5178
  threshold=0.433 → F1=0.5299 (optimal)
  AUC-ROC=0.7345   Gini=0.4691
              precision    recall  f1-score   support

  No Default     0.8642    0.5376    0.6629      1077
     Default     0.4000    0.7849    0.5299       423

    accuracy                         0.6073      1500
   macro avg     0.6321    0.6612    0.5964      1500
weighted avg     0.7333    0.6073    0.6254      1500



In [48]:
# LightGBM
print('=== LightGBM ===')
t0 = time.time()
lgbm = lgb.LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.07,
                            num_leaves=40, min_child_samples=20,
                            scale_pos_weight=pos_weight,
                            random_state=SEED, n_jobs=-1, verbose=-1)
lgbm.fit(X_tr, y_tr)
r = evaluate_model('LightGBM (library)', lgbm.predict_proba(X_val)[:,1], y_val)
r.update({'train_sec': round(time.time()-t0,2), 'model':lgbm, 'scaled':False})
results.append(r)


=== LightGBM ===
  LightGBM (library)
  threshold=0.500 → F1=0.5016
  threshold=0.449 → F1=0.5198 (optimal)
  AUC-ROC=0.7347   Gini=0.4695
              precision    recall  f1-score   support

  No Default     0.8266    0.6815    0.7471      1077
     Default     0.4395    0.6359    0.5198       423

    accuracy                         0.6687      1500
   macro avg     0.6331    0.6587    0.6334      1500
weighted avg     0.7174    0.6687    0.6830      1500



## Feature Selection via RF Feature Importance

After training Random Forest, we rank features by importance and remove  
those below threshold **= 0.01** (less than 1% contribution).

This step:
- Prevents noisy features from degrading model quality
- Reduces dimensionality for Custom GBM (faster training)
- Provides interpretable signal about which features matter


In [49]:
fi = pd.Series(rf.feature_importances_, index=X_tr.columns).sort_values(ascending=False)
print('Feature Importance (RF):')
print(fi.round(4).to_string())

IMPORTANCE_THRESHOLD = 0.01
low_fi = fi[fi < IMPORTANCE_THRESHOLD].index.tolist()
print(f'\nFeatures below threshold {IMPORTANCE_THRESHOLD}: {low_fi}')

if low_fi:
    X_tr2  = X_tr.drop(columns=low_fi)
    X_val2 = X_val.drop(columns=low_fi)
    X_te2  = X_te.drop(columns=low_fi)
    print(f'After feature selection: {X_tr2.shape[1]} features remain')
else:
    X_tr2, X_val2, X_te2 = X_tr.copy(), X_val.copy(), X_te.copy()

# Plot
fig, ax = plt.subplots(figsize=(9, 7))
fi.sort_values().plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.axvline(IMPORTANCE_THRESHOLD, color='red', ls='--', alpha=0.7,
           label=f'Selection threshold = {IMPORTANCE_THRESHOLD}')
ax.set_title('Feature Importance (RF) + Feature Selection Threshold', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT}/05_feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()


Feature Importance (RF):
Credit Score                 0.2735
Log_Loan                     0.1616
Log_Income                   0.1042
DTI                          0.0964
Term                         0.0827
Monthly Debt                 0.0557
Log_MaxCred                  0.0553
Credit_Utilization           0.0537
Years of Credit History      0.0449
Number of Open Accounts      0.0238
Years in current job         0.0176
Home Ownership               0.0138
Purpose                      0.0075
Has_Delinquent               0.0064
Number of Credit Problems    0.0025
Tax Liens                    0.0003

Features below threshold 0.01: ['Purpose', 'Has_Delinquent', 'Number of Credit Problems', 'Tax Liens']
After feature selection: 12 features remain


## Part 7. ⭐ Custom Gradient Boosting — Hand-Written Implementation

### Algorithm (Friedman, 2001)

**Loss function:** Binary Cross-Entropy  
$$L(y, p) = -[y \cdot \log(p) + (1-y) \cdot \log(1-p)]$$

**Pseudo-residuals** (negative gradient of loss):  
$$r_i = y_i - p_i, \quad p_i = \sigma(F(x_i)) = \frac{1}{1+e^{-F(x_i)}}$$

**Update rule** at each iteration m:  
$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

where $h_m$ is a `DecisionTreeRegressor` trained on pseudo-residuals $r$.

**Key parameters:**
- `n_estimators` — number of boosting rounds
- `learning_rate` η — shrinks each tree's contribution (prevents overfitting)
- `max_depth` — depth of base trees
- `subsample` — fraction of data sampled per iteration (stochastic gradient boosting)


In [50]:
class CustomGradientBoostingClassifier:
    '''
    Hand-written Gradient Boosting for binary classification.
    Based on Friedman (2001) — Greedy Function Approximation: A Gradient Boosting Machine.

    Algorithm:
      1. Initialise F0 = log(p_mean / (1 - p_mean))  [log-odds of base rate]
      2. For m = 1..M:
         a. Compute probabilities: p = sigmoid(F)
         b. Compute pseudo-residuals: r = y - p  [negative gradient of log-loss]
         c. Subsample indices (stochastic GBM)
         d. Fit DecisionTreeRegressor on (X_sub, r_sub)
         e. Update: F += learning_rate * tree.predict(X)
      3. Final prediction: P(y=1) = sigmoid(F)
    '''

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=4,
                 min_samples_leaf=10, subsample=0.8, random_state=42):
        self.n_estimators     = n_estimators
        self.learning_rate    = learning_rate
        self.max_depth        = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.subsample        = subsample
        self.random_state     = random_state
        self.trees_           = []
        self.F0_              = None
        self.train_loss_      = []

    @staticmethod
    def _sigmoid(x):
        '''Numerically stable sigmoid function.'''
        return np.where(x >= 0,
                        1 / (1 + np.exp(-x)),
                        np.exp(x) / (1 + np.exp(x)))

    @staticmethod
    def _log_loss(y, prob):
        '''Binary cross-entropy loss.'''
        eps = 1e-15
        prob = np.clip(prob, eps, 1 - eps)
        return -np.mean(y * np.log(prob) + (1 - y) * np.log(1 - prob))

    def fit(self, X, y):
        X = np.array(X, dtype=np.float64)
        y = np.array(y, dtype=np.float64)
        rng = np.random.RandomState(self.random_state)
        n_samples = len(y)

        # Step 0: initial prediction = log-odds
        p_mean = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        self.F0_ = np.log(p_mean / (1 - p_mean))
        F = np.full(n_samples, self.F0_, dtype=np.float64)

        for m in range(self.n_estimators):
            # Step 1: current probabilities
            prob = self._sigmoid(F)
            # Step 2: pseudo-residuals = negative gradient
            residuals = y - prob
            # Step 3: stochastic subsampling
            n_sub = max(1, int(self.subsample * n_samples))
            idx   = rng.choice(n_samples, size=n_sub, replace=False)
            # Step 4: fit base learner on residuals
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                random_state=self.random_state)
            tree.fit(X[idx], residuals[idx])
            self.trees_.append(tree)
            # Step 5: update F
            F += self.learning_rate * tree.predict(X)
            # Track loss
            self.train_loss_.append(self._log_loss(y, self._sigmoid(F)))

        return self

    def predict_proba(self, X):
        X = np.array(X, dtype=np.float64)
        F = np.full(len(X), self.F0_, dtype=np.float64)
        for tree in self.trees_:
            F += self.learning_rate * tree.predict(X)
        prob_pos = self._sigmoid(F)
        return np.column_stack([1 - prob_pos, prob_pos])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

    def get_feature_importances(self, feature_names=None):
        '''Average feature importance across all trees.'''
        n_feat = self.trees_[0].n_features_in_
        imp = np.zeros(n_feat)
        for tree in self.trees_:
            imp += tree.feature_importances_
        imp /= len(self.trees_)
        if feature_names is not None:
            return pd.Series(imp, index=feature_names).sort_values(ascending=False)
        return imp

print('CustomGradientBoostingClassifier defined ✓')


CustomGradientBoostingClassifier defined ✓


In [51]:
# Train Custom GBM
print('Training CustomGBM...')
t0 = time.time()
custom_gbm = CustomGradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.07,
    max_depth=4,
    min_samples_leaf=15,
    subsample=0.8,
    random_state=SEED
)
custom_gbm.fit(np.array(X_tr2), y_tr.values)
custom_time = round(time.time() - t0, 2)
print(f'Custom GBM trained in {custom_time}s')

prob_custom = custom_gbm.predict_proba(np.array(X_val2))[:, 1]
r_custom = evaluate_model('Custom GBM (hand-written)', prob_custom, y_val)
r_custom.update({'train_sec': custom_time, 'model': custom_gbm, 'scaled': False})
results.append(r_custom)


Training CustomGBM...
Custom GBM trained in 3.21s
  Custom GBM (hand-written)
  threshold=0.500 → F1=0.3835
  threshold=0.301 → F1=0.5290 (optimal)
  AUC-ROC=0.7386   Gini=0.4772
              precision    recall  f1-score   support

  No Default     0.8234    0.7447    0.7821      1077
     Default     0.4772    0.5934    0.5290       423

    accuracy                         0.7020      1500
   macro avg     0.6503    0.6690    0.6555      1500
weighted avg     0.7258    0.7020    0.7107      1500



In [52]:
# Learning curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(custom_gbm.train_loss_)+1), custom_gbm.train_loss_,
        color='#e74c3c', linewidth=1.8)
ax.set_xlabel('Iteration (n_estimators)', fontsize=11)
ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=11)
ax.set_title('Custom GBM — Training Loss Curve', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/06_custom_gbm_learning_curve.png', dpi=130, bbox_inches='tight')
plt.show()

# Feature Importance
fi_custom = custom_gbm.get_feature_importances(feature_names=list(X_tr2.columns))
print('\nTop-10 features (Custom GBM):')
print(fi_custom.head(10).round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
fi_custom.sort_values().plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Feature Importance — Custom GBM (averaged over trees)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/07_custom_gbm_feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()



Top-10 features (Custom GBM):
Credit Score               0.3282
Log_Loan                   0.1720
Log_Income                 0.1651
DTI                        0.1321
Term                       0.1119
Log_MaxCred                0.0298
Credit_Utilization         0.0247
Years of Credit History    0.0141
Home Ownership             0.0086
Monthly Debt               0.0076


## Part 8. Comparison: Library vs Custom GBM

Training LightGBM with **identical parameters and data** for a fair comparison.


In [53]:
# LightGBM with same params as Custom GBM
t0 = time.time()
lgbm2 = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.07,
                             num_leaves=31, min_child_samples=15,
                             scale_pos_weight=pos_weight,
                             random_state=SEED, n_jobs=-1, verbose=-1)
lgbm2.fit(X_tr2, y_tr)
lgbm2_time = round(time.time()-t0, 2)

prob_lgbm2 = lgbm2.predict_proba(X_val2)[:,1]
r_lgbm2 = evaluate_model('LightGBM (same params)', prob_lgbm2, y_val)
r_lgbm2.update({'train_sec': lgbm2_time, 'model': lgbm2, 'scaled': False})
results.append(r_lgbm2)

print('\n=== Direct Comparison (same data + same params) ===')
print(f'{"Model":<30} {"F1(tuned)":>10} {"AUC":>8} {"Gini":>8} {"Time(s)":>10}')
print('─'*72)
for row in [r_custom, r_lgbm2]:
    print(f'  {row["name"]:<28} {row["f1_tuned"]:>10.4f} {row["auc"]:>8.4f} '
          f'{row["gini"]:>8.4f} {row["train_sec"]:>9.2f}s')
print(f'\nSpeed ratio: LightGBM is {lgbm2_time/max(custom_time,0.01):.1f}x faster')
print(f'F1 gap: {abs(r_lgbm2["f1_tuned"]-r_custom["f1_tuned"]):.4f}')
print(f'Gini gap: {abs(r_lgbm2["gini"]-r_custom["gini"]):.4f}')


  LightGBM (same params)
  threshold=0.500 → F1=0.5115
  threshold=0.371 → F1=0.5252 (optimal)
  AUC-ROC=0.7411   Gini=0.4821
              precision    recall  f1-score   support

  No Default     0.8969    0.4280    0.5795      1077
     Default     0.3753    0.8747    0.5252       423

    accuracy                         0.5540      1500
   macro avg     0.6361    0.6514    0.5524      1500
weighted avg     0.7498    0.5540    0.5642      1500


=== Direct Comparison (same data + same params) ===
Model                           F1(tuned)      AUC     Gini    Time(s)
────────────────────────────────────────────────────────────────────────
  Custom GBM (hand-written)        0.5290   0.7386   0.4772      3.21s
  LightGBM (same params)           0.5252   0.7411   0.4821      0.43s

Speed ratio: LightGBM is 0.1x faster
F1 gap: 0.0038
Gini gap: 0.0049


In [54]:
# Full comparison table
rdf = pd.DataFrame([{k:v for k,v in r.items() if k not in ('model','scaled')} for r in results])
rdf = rdf.sort_values('f1_tuned', ascending=False).reset_index(drop=True)
rdf.to_csv(f'{OUT}/model_comparison.csv', index=False)
print('All models comparison:')
rdf[['name','f1_default','f1_tuned','threshold','auc','gini','train_sec']]


All models comparison:


,name,f1_default,f1_tuned,threshold,auc,gini,train_sec
0,RF (library),0.5178,0.5299,0.433,0.7345,0.4691,1.16
1,Custom GBM (hand-written),0.3835,0.5290,0.301,0.7386,0.4772,3.21
2,LightGBM (same params),0.5115,0.5252,0.371,0.7411,0.4821,0.43
3,LightGBM (library),0.5016,0.5198,0.449,0.7347,0.4695,0.39
4,DT (library),0.4821,0.4975,0.369,0.7002,0.4004,0.12
5,LR (library),0.4586,0.4774,0.400,0.6503,0.3006,0.04


In [55]:
# Comparison plot
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
metrics = [('f1_tuned','F1 (optimal threshold)'), ('auc','AUC-ROC'), ('gini','Gini (2×AUC−1)')]
clr_map = {'Custom GBM':'#e74c3c','LightGBM':'#f39c12','RF':'#27ae60',
           'DT':'#3498db','LR':'#9b59b6'}
for ax, (m, title) in zip(axes, metrics):
    vals  = rdf[m].values
    names = rdf['name'].str.replace(' (', '\n(', regex=False).values
    clrs  = ['#e74c3c' if 'Custom' in n else '#f39c12' if 'LightGBM' in n else
             '#27ae60' if 'RF' in n else '#3498db' if 'DT' in n else '#9b59b6'
             for n in rdf['name'].values]
    bars = ax.barh(names, vals, color=clrs, edgecolor='white', height=0.6)
    for b, v in zip(bars, vals):
        ax.text(v+0.003, b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
    if m == 'f1_tuned':
        ax.axvline(0.5, color='black', ls='--', alpha=0.5, label='0.5 target')
        ax.legend(fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, min(1.12, max(vals)+0.14))
    ax.tick_params(labelsize=8)
plt.suptitle('Model Comparison: Library vs Custom GBM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/08_model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()


In [56]:
# Confusion matrix for best model
best = max(results, key=lambda r: r['f1_tuned'])
print(f'Best model: {best["name"]}  F1={best["f1_tuned"]:.4f}  Gini={best["gini"]:.4f}')

bm = best['model']
if isinstance(bm, CustomGradientBoostingClassifier):
    prob_b = bm.predict_proba(np.array(X_val2))[:,1]
elif 'same' in best['name']: prob_b = lgbm2.predict_proba(X_val2)[:,1]
elif best.get('scaled'): prob_b = bm.predict_proba(X_val_s)[:,1]
else: prob_b = bm.predict_proba(X_val)[:,1]
pred_b = (prob_b >= best['threshold']).astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_val, pred_b),
                       display_labels=['No Default','Default']).plot(ax=ax, cmap='Blues')
ax.set_title(f"Confusion Matrix — {best['name']}\nthr={best['threshold']:.3f}", fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/09_confusion_matrix.png', dpi=130, bbox_inches='tight')
plt.show()


Best model: RF (library)  F1=0.5299  Gini=0.4691


## Part 9. Predictions for Test Set

In [57]:
# Use LightGBM (same params) as final model
prob_test = lgbm2.predict_proba(X_te2)[:,1]
pred_test = (prob_test >= r_lgbm2['threshold']).astype(int)

sub = pd.DataFrame({'id': range(len(pred_test)),
                    'Credit Default': pred_test,
                    'Default_Probability': prob_test.round(4)})
sub.to_csv(f'{OUT}/predictions.csv', index=False)
print(f'Saved: predictions.csv ({len(sub)} rows)')
print(f'Default rate in test: {pred_test.mean()*100:.1f}%')
sub.head(10)


Saved: predictions.csv (2500 rows)
Default rate in test: 64.5%


,id,Credit Default,Default_Probability
0,0,1,0.5738
1,1,1,0.5633
2,2,1,0.8801
3,3,1,0.4616
4,4,1,0.4436
5,5,1,0.3959
6,6,1,0.4291
7,7,1,0.4600
8,8,1,0.6013
9,9,0,0.1704


## Final Summary

In [58]:
print('='*60)
print('FINAL RESULTS')
print('='*60)
print(rdf[['name','f1_tuned','gini','train_sec']].to_string(index=False))
print()
best_lib = max([r for r in results if 'Custom' not in r['name'] and 'same' not in r['name']],
               key=lambda r: r['f1_tuned'])
print(f'Best library model : {best_lib["name"]}')
print(f'  F1 (tuned)       : {best_lib["f1_tuned"]:.4f}  {"✓ > 0.5" if best_lib["f1_tuned"]>0.5 else "✗"}')
print(f'  Gini             : {best_lib["gini"]:.4f}')
print()
print(f'Custom GBM (⭐ hand-written):')
print(f'  F1 (tuned)       : {r_custom["f1_tuned"]:.4f}')
print(f'  Gini             : {r_custom["gini"]:.4f}')
print(f'  Train time       : {r_custom["train_sec"]:.2f}s')
print(f'  vs LightGBM F1   : {r_lgbm2["f1_tuned"]:.4f} (gap={abs(r_lgbm2["f1_tuned"]-r_custom["f1_tuned"]):.4f})')
print()
print('Output files:')
for f in sorted(glob.glob(f'{OUT}/*')):
    print(f'  {os.path.basename(f):<45} {os.path.getsize(f)//1024:>4} KB')


FINAL RESULTS
                     name  f1_tuned   gini  train_sec
             RF (library)    0.5299 0.4691       1.16
Custom GBM (hand-written)    0.5290 0.4772       3.21
   LightGBM (same params)    0.5252 0.4821       0.43
       LightGBM (library)    0.5198 0.4695       0.39
             DT (library)    0.4975 0.4004       0.12
             LR (library)    0.4774 0.3006       0.04

Best library model : RF (library)
  F1 (tuned)       : 0.5299  ✓ > 0.5
  Gini             : 0.4691

Custom GBM (⭐ hand-written):
  F1 (tuned)       : 0.5290
  Gini             : 0.4772
  Train time       : 3.21s
  vs LightGBM F1   : 0.5252 (gap=0.0038)

Output files:
  01_class_imbalance.png                          25 KB
  02_distributions_by_class.png                   87 KB
  03_outliers_boxplot.png                         95 KB
  04_correlation_matrix.png                      270 KB
  05_feature_importance.png                       63 KB
  06_custom_gbm_learning_curve.png                48 KB
  0